In [32]:
import os
from haystack import Document
from haystack.document_stores.types import DuplicatePolicy
from haystack_integrations.document_stores.faiss import FAISSDocumentStore
from sentence_transformers import SentenceTransformer
import json
import pandas as pd
from haystack import Document
from haystack.document_stores.in_memory import InMemoryDocumentStore
import pickle
import numpy as np
from pathlib import Path
print(os.getcwd())

/Users/2649949M/Library/CloudStorage/OneDrive-UniversityofGlasgow/Desktop/Reproduction/fair_rag


In [33]:
def load_test_collections(test_folder=None, new_folder =None, test_file=None):
    if test_file:
        paths = [Path(test_file)]
    else:
        paths = sorted(Path(test_folder).glob("*.csv"))

    if not paths:
        raise FileNotFoundError("No test CSV files were found.")

    collections = []
    for path in paths:
        
        frame = pd.read_csv(path)
        frame.rename(columns={'gender':"gender_id", 'gender_string':'gender'})
        frame.to_csv(new_folder+str(path.name))

        
    return collections

In [35]:
collection = load_test_collections(test_folder="datasets/occupations_full/", new_folder="datasets/occupations/")

In [2]:
import os

os.environ["HF_HUB_DISABLE_XET"] = "1"

from sentence_transformers import SentenceTransformer

model = SentenceTransformer("BAAI/bge-large-en")
print(model.get_sentence_embedding_dimension())

1024


In [3]:
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained(
    "BAAI/bge-large-en"
)

print("success")

success


In [2]:
print("hi")

wiki_path = "datasets/wiki.pkl"
mapping_path = "datasets/occupation mapping.csv"
file_path = "datasets/"

hi


In [3]:
wiki = pd.read_csv("datasets/wiki.csv")
print(wiki.columns)

Index(['Unnamed: 0', 'wikidata_code', 'gender', 'birth', 'birth_estimation',
       'birth_era', 'death', 'death_estimation', 'death_era',
       'citizenship_primary', 'citizenship_secondary', 'un_region',
       'un_subregion', 'birthplace_lon', 'birthplace_lat', 'deathplace_lon',
       'deathplace_lat', 'occupation_level_1', 'occupation_level_2',
       'occupation_level_3', 'number_wiki_editions', 'total_word_count',
       'visibility_ranking', 'visibility_sum', 'wiki_readers_2015_2018',
       'curid', 'text', 'name', 'wikipedia_url', 'title', 'id'],
      dtype='str')


In [4]:
wiki_names = wiki["name"]
wiki_title = wiki['title']

In [10]:
print(wiki_names.shape)
print(wiki_names.nunique)

<bound method IndexOpsMixin.nunique of 0                  Chris_Wilkerson
1                    Nedra_Pickler
2                Professor_Lyrical
3                      Niki_Reiser
4                Andrew_Fitzgerald
                    ...           
1434878           Fabrizio_Anselmi
1434879             Rodolfo_Soares
1434880    Gabriel_Christoni_Leite
1434881        Emiliano_Pattarello
1434882           Dario_D'Ambrosio
Name: name, Length: 1434883, dtype: str>


In [11]:
print(wiki_title.shape)
print(wiki_title.nunique)

(1434883,)
<bound method IndexOpsMixin.nunique of 0                                Chris Wilkerson
1                                  Nedra Pickler
2                              Professor Lyrical
3                                    Niki Reiser
4                              Andrew Fitzgerald
                           ...                  
1434878                         Fabrizio Anselmi
1434879                           Rodolfo Soares
1434880    Gabriel Leite (footballer, born 1987)
1434881                      Emiliano Pattarello
1434882                         Dario D'Ambrosio
Name: title, Length: 1434883, dtype: str>


In [18]:
wiki_name_title = wiki[["name", "title"]]
test = wiki_name_title[wiki_name_title["name"]==wiki_name_title['title']]


In [21]:
print(test.shape)
print(wiki_name_title.shape)

(11795, 2)
(1434883, 2)


In [15]:
wiki_24 = pd.read_csv("datasets/wiki_24.csv")

In [16]:
wiki_24_plus_title = wiki_24.merge(wiki_name_title,on="name", how='left' )

In [27]:
print(wiki_24_plus_title.shape)
test = wiki_24_plus_title[wiki_24_plus_title["title"]==None]

(540106, 5)


In [29]:
wiki_24_name_title = wiki_24_plus_title[["name", "title"]]

In [30]:
wiki_24_plus_title.to_csv("datasets/name_title.csv")

In [28]:
print(test.shape)


(0, 5)


In [17]:
print(wiki_24_plus_title)

                                                     text  gender  \
0       Nedra J. Pickler (born October 13, 1975) is an...  Female   
1       cover national political issues in December 20...  Female   
2       and public schools. On March 27, 2007, Pickler...  Female   
3       work as a managing director at The Glover Park...  Female   
4       Niki Reiser (born 12 May 1958) is a Swiss film...    Male   
...                                                   ...     ...   
540101  Amar-te Mais (2013) Crónica da Selva (2014) O ...    Male   
540102  Marcos Hummel (born July 25, 1947) is a Brazil...    Male   
540103  "Journal da Manchete" and "Na Linha do Crime"....    Male   
540104  Armando Nogueira (January 27, 1927 – March 29,...    Male   
540105  Leandro Okabe de Oliveira Bulhões, born on Jun...    Male   

                    name  occupation             title  
0          Nedra_Pickler  journalist     Nedra Pickler  
1          Nedra_Pickler  journalist     Nedra Pickler  


In [3]:
wiki = pd.read_pickle(wiki_path)


In [7]:
wiki.to_pickle(file_path+"wiki.pkl")

In [6]:
print(wiki.size)

43046490


In [6]:
import numpy as np
print(np.__version__)

1.26.4


In [5]:
def preprocess_people_in_wikipedia(wiki_path, mapping_path, file_path):
    wiki = pd.read_pickle(wiki_path)
    mapping = pd.read_csv(mapping_path)
    occupation_list = set(mapping['profession_name'])
    max_tokens =150
    passages = []
    for _, row in wiki .iterrows():
        if row['occupation_level_3'] in occupation_list:
            words = row['text'].split()
            for start in range(0, len(words), max_tokens):
                passage_text = " ".join(words[start:start + max_tokens])
                passages.append({
                    'text': passage_text,
                    'gender': row['gender'],
                    'name': row['name'],
                'occupation': row['occupation_level_3']
            })
    wiki_subset = pd.DataFrame(passages)
    wiki_subset.to_csv(file_path+"wiki_24.csv", index=False)
    return wiki_subset

In [ ]:
def create_sparse_index(wiki_df, save_path):
    docs = [Document(content=row["text"], meta = {"gender":row["gender"], "title":row["title"], "occupation":row["occupation"]}) for _, row in wiki_df.iterrows()]
    document_store = InMemoryDocumentStore(bm25_algorithm="BM25Okapi")

    document_store.write_documents(docs)
    with open(save_path, "wb") as f:
        pickle.dump(document_store, f)
    return document_store

In [ ]:
def create_dense_index(wiki_df, index_path="/datasets/wiki_people"):
    #model = SentenceTransformer("BAAI/bge-large-en")
    #texts = wiki_df["text"].tolist()
    #embeddings = model.encode(texts, batch_size=256, show_progress_bar = True, convert_to_numpy=True, normalize_embeddings=True)
    embeddings = np.load('datasets/wiki_embeddings.npy')
    docs = []
    print("encoded")
    for (_,row), embedding in zip(wiki_df.iterrows(), embeddings):
        docs.append(Document(content=row["text"],embedding=embedding, meta={"gender":row["gender"], "name":row["name"], "occupation":row["occupation"]}))
    document_store = FAISSDocumentStore(index_path=index_path, index_string ="HNSW32", embedding_dim=embeddings.shape[1])
    document_store.write_documents(docs, policy=DuplicatePolicy.OVERWRITE,)
    document_store.save(index_path)
    return document_store


: 

In [ ]:
import os
wiki_df = pd.read_csv(file_path+"wiki_24.csv")#preprocess_people_in_wikipedia(wiki_path=wiki_path, mapping_path=mapping_path, file_path=file_path)
os.environ["HF_HUB_DISABLE_XET"] = "1"

doc = create_dense_index(wiki_df=wiki_df)

encoded


In [4]:
from huggingface_hub import hf_hub_download
import pandas as pd
repod_id = "LabHC/bias_in_bios"
file_name = "bios_dev.pkl"

dataset = pd.read_pickle(hf_hub_download(repo_id=repod_id, filename=file_name,repo_type="dataset"))

In [5]:
print(dataset.size)

118926


In [10]:
print(dataset.memory_usage(index=True).sum())

2377788


In [6]:
dataset.to_csv("bias_in_bios_dev.csv")

In [12]:
from nomic import AtlasDataset

dataset = AtlasDataset("nomic/wiki-people")

2026-10-13 16:36:44.547 | INFO     | nomic.dataset:__init__:817 - Loading existing dataset `nomic/wiki-people`.


In [13]:
atlas_map = dataset.maps[0]
df = atlas_map.data.df

2026-10-13 16:39:54.455 | WARNING  | nomic.data_operations:df:1173 - Converting to pandas dataframe. This may materialize a large amount of data into memory.
2026-10-13 16:39:54.456 | INFO     | nomic.data_operations:_download_data:1142 - Downloading data
100%|██████████| 69/69 [00:00<00:00, 14603.24it/s]
2026-10-13 16:41:52.281 | INFO     | nomic.data_operations:_load_data:1069 - Loading data
100%|██████████| 69/69 [00:02<00:00, 24.15it/s]


In [16]:
df.head

<bound method NDFrame.head of         wikidata_code  gender   birth  birth_estimation  \
0           Q16983445    Male     NaN               NaN   
1            Q6986310  Female  1975.0            1975.0   
2           Q16734751    Male     NaN               NaN   
3             Q332694    Male  1958.0            1958.0   
4            Q4756960    Male  1763.0            1763.0   
...               ...     ...     ...               ...   
1434878      Q5428194    Male  1978.0            1978.0   
1434879      Q7357199    Male  1985.0            1985.0   
1434880      Q5515571    Male  1987.0            1987.0   
1434881     Q57242454    Male  1999.0            1999.0   
1434882      Q5222836    Male  1988.0            1988.0   

                                 birth_era   death  death_estimation  \
0                                  Missing     NaN               NaN   
1        5.Contemporary period 1901-2020AD     NaN       2056.805664   
2                                  Missing   

In [17]:
df.to_pickle("wiki.pkl")

In [27]:
import pandas as pd
import os


In [14]:
print(os.getcwd())

/Users/2649949M/Desktop/Reproduction/fair_rag


In [2]:
import pandas as pd
b_in_b = pd.read_csv("datasets/bias_in_bios_train.csv")

In [2]:
import pandas as pd

In [28]:
wiki_24 = pd.read_pickle("datasets/wiki.pkl")

In [29]:
print(wiki_24.shape[0])

1434883


In [10]:
wiki_1st_row = wiki_24.iloc[0]

In [11]:
print(wiki_1st_row["text"])

Chris Wilkerson is an American college football coach. He is the head football coach at Eastern Illinois University a position he assumed in 2022, succeeding Adam Cushing. He served as the head football coach at the University of Chicago from 2013 to 2021. A graduate of Eastern Illinois, Wilkerson was an assistant football coach at his alma mater as well as San Jose State University and Dartmouth College.

Head coaching record


In [3]:
mapping = pd.read_csv("datasets/occupation mapping.csv")

In [4]:
b_in_b = b_in_b.merge(mapping,on="profession", how='left' )

In [28]:
b_in_b_pl = b_in_b[b_in_b["profession_name"].str.contains("paralegal")]

In [5]:
print(b_in_b_pl.head())

NameError: name 'b_in_b_pl' is not defined

In [6]:
bios_jobs = b_in_b["profession_name"].unique().tolist()

In [42]:
for job in bios_jobs:
    job_rows = b_in_b.loc[b_in_b["profession_name"] == job]
    texts = job_rows["hard_text"].tolist()
    wiki_job_rows = wiki_24.loc[wiki_24["occupation_level_3"] == job]
    wiki_texts = wiki_job_rows["text"].tolist()
    print(len(wiki_texts))
    for text in texts:
        len_text = len(text)
        for txt in wiki_texts:
            len_txt = len(txt)
            if len_txt == len_text:
                list_of_words_text = text.split()
                list_of_words_txt = txt.split()
                output = [x for x in list_of_words_text if not x in list_of_words_txt or list_of_words_txt.remove(x)]
                if len(output) == 0:
                    output = "The two texts are identical"
                    print(output) 
             


               
                
            

3783
18148
2208
8980
193
611
7492
6789
2993
104
489
2180
309
18435
772
0
28938
2005
3875
15784
0
2
0
0
760
15
1727
17


In [27]:
print(wiki_comedian.head())

       wikidata_code  gender   birth  birth_estimation  \
371426      Q6396570    Male  1944.0            1944.0   
822660     Q23016625  Female  1912.0            1912.0   

                                birth_era   death  death_estimation  \
371426  5.Contemporary period 1901-2020AD     NaN       2021.794434   
822660  5.Contemporary period 1901-2020AD  1997.0       1997.000000   

                                death_era citizenship_primary  \
371426                            Missing                  US   
822660  5.Contemporary period 1901-2020AD              Mexico   

       citizenship_secondary  ... total_word_count visibility_ranking  \
371426                  null  ...           2664.0          1781916.0   
822660                    US  ...           1593.0          2473840.5   

        visibility_sum  wiki_readers_2015_2018       curid  \
371426       17.014048                  4592.0  27382744.0   
822660       14.939277                   961.0  49650792.0   

        

In [21]:
print(b_in_b.loc[2])

Unnamed: 0                                                         2
hard_text          She is averse to all things scary or sad, so s...
profession                                                         4
gender                                                             1
profession_name                                             comedian
Name: 2, dtype: object


In [4]:
b_in_b['profession'].max()

np.int64(27)

In [7]:
mapping = pd.read_csv("occupation mapping.csv")

FileNotFoundError: [Errno 2] No such file or directory: 'occupation mapping.csv'

In [8]:
mapping.columns
b_in_b.columns

Index(['Unnamed: 0', 'hard_text', 'profession', 'gender', 'profession_name'], dtype='str')

In [8]:
wiki = pd.read_pickle("wiki.pkl")

In [9]:
wiki['occupation_level_1'].head

<bound method NDFrame.head of 0               Sports/Games
1                    Culture
2                    Culture
3                    Culture
4          Discovery/Science
                 ...        
1434878         Sports/Games
1434879         Sports/Games
1434880         Sports/Games
1434881         Sports/Games
1434882         Sports/Games
Name: occupation_level_1, Length: 1434883, dtype: category
Categories (6, str): ['Sports/Games', 'Leadership', 'Other', 'Culture', 'Discovery/Science', 'Missing']>

In [10]:
wiki["occupation_level_2"]

0               Sports/Games
1          Culture-periphery
2               Culture-core
3               Culture-core
4                   Academia
                 ...        
1434878         Sports/Games
1434879         Sports/Games
1434880         Sports/Games
1434881         Sports/Games
1434882         Sports/Games
Name: occupation_level_2, Length: 1434883, dtype: category
Categories (15, str): ['Sports/Games', 'Religious', 'Worker/Business (small)', 'Culture-core', ..., 'Explorer/Inventor/Developer', 'Missing', 'Family', 'Other']

In [11]:
occ = wiki["occupation_level_3"]

In [29]:
test = occ.str.contains('_master', regex=True)

In [30]:
test[test].index

Index([  99946,  308500,  432798,  441745,  443847,  448787,  457741,  460038,
        502677, 1068130, 1087447, 1091472, 1122746, 1147071, 1161462, 1174929,
       1183513],
      dtype='int64')

In [31]:
occ[test]

99946         tea_master
308500     bonsai_master
432798        tea_master
441745       sufi_master
443847     bonsai_master
448787       sufi_master
457741        tea_master
460038        tea_master
502677       sufi_master
1068130    bonsai_master
1087447      sufi_master
1091472       tea_master
1122746      sufi_master
1147071      sufi_master
1161462      sufi_master
1174929      yoga_master
1183513      yoga_master
Name: occupation_level_3, dtype: category
Categories (2507, str): ['swimmer', 'football', 'priest', 'worker', ..., 'criminale', 'orthopedist', 'califa', 'minatore']

In [18]:
import pandas as pd

b_in_b_train = pd.read_csv("datasets/bias_in_bios_dev.csv")

In [19]:
mapping = pd.read_csv("datasets/occupation mapping.csv")

In [20]:
mapping.columns

Index(['profession', 'profession_name'], dtype='str')

In [9]:
b_in_b.columns

Index(['Unnamed: 0', 'hard_text', 'profession', 'gender', 'profession_name'], dtype='str')

In [21]:
b_in_b = b_in_b_train.merge(mapping,on="profession", how='left' )

In [22]:
b_in_b.head

<bound method NDFrame.head of        Unnamed: 0                                          hard_text  \
0               0  Tampio researches the history of political tho...   
1               1  He has been a practicing Dentist for 11 years....   
2               2  Mr. Miserez devotes a substantial portion of h...   
3               3  She holds a master’s degree in nutrition educa...   
4               4  She received her Ph.D. in 2013 from Florida In...   
...           ...                                                ...   
39637       39637  Dr. Danish Khan practices at Dent O Care Denta...   
39638       39638  He is one of the oldest readers of the Eye of ...   
39639       39639  Born in Barranquilla, Colombia, Stephanie has ...   
39640       39640  Trained as a documentary storyteller as a film...   
39641       39641  Ms. Monaco is a named SuperLawyer®. She is a m...   

       profession  gender profession_name  
0              21       0       professor  
1               6

In [10]:
counts =  pd.crosstab(b_in_b["profession_name"],b_in_b['gender'])

In [24]:
print(b_in_b['hard_text'].str.len().min())

150


In [25]:
count2 = pd.crosstab(b_in_b["profession"], b_in_b["gender"])
print(count2)

gender         0     1
profession            
0            357   207
1            772   240
2           2010  1248
3            196    70
4            222    60
5            469    92
6            944   515
7             29   367
8            128    22
9            470   231
10            28   118
11          1007   988
12           130   620
13           174  1722
14           420   354
15            27   150
16           192    61
17            78    66
18          1561   867
19          2076  2024
20           358   344
21          6482  5326
22           698  1141
23           127    14
24           582   110
25          1158   202
26           645   976
27            26   141


In [11]:
print(counts)

gender                 0      1
profession_name                
accountant          2317   1343
architect           5014   1554
attorney           13064   8105
chiropractor        1271    454
comedian            1439    385
composer            3042    595
dentist             6133   3346
dietitian            183   2384
dj                   828    136
filmmaker           3048   1497
interior_designer    182    767
journalist          6545   6415
model                840   4027
nurse               1127  11189
painter             2727   2298
paralegal            173    973
pastor              1245    393
personal_trainer     505    423
photographer       10141   5632
physician          13492  13156
poet                2323   2235
professor          42130  34618
psychologist        4530   7415
rapper               823     88
software_engineer   3783    709
surgeon             7521   1308
teacher             4188   6343
yoga_teacher         166    910


In [13]:
print(counts.columns)

Index([0, 1], dtype='int64', name='gender')


In [22]:
print(counts[counts[1]<=100])
print(counts[counts[0]<=100])
print('heree')
count_new = counts[(counts[1]<=100) | (counts[0]<=100)]
print(count_new)


gender             0   1
profession_name         
rapper           823  88
Empty DataFrame
Columns: [0, 1]
Index: []
heree
gender             0   1
profession_name         
rapper           823  88


In [23]:
print(count_new.index.to_list())

['rapper']


In [24]:
import haystack

print(haystack.__version__)

3.4.0


In [3]:
wiki = pd.read_pickle('wiki.pkl')

In [27]:
from haystack import Document
from haystack.components.retrievers.in_memory import InMemoryBM25Retriever
from haystack.document_stores.in_memory import InMemoryDocumentStore

docs = [
    Document(content="Python is a popular programming language", meta={"source": "wiki"}),
    Document(content="python ist eine beliebte Programmiersprache", meta={"source": "wiki"}),
]

doc_store = InMemoryDocumentStore()
doc_store.write_documents(docs)
retriever = InMemoryBM25Retriever(doc_store)

result = retriever.run(query="Programmiersprache")

print(result["documents"])

[Document(id=2d6e811b00963ea842d233380ab501d1d66a2cc789b4dab81614f84f91adbfca, content: 'python ist eine beliebte Programmiersprache', meta: {'source': 'wiki'}, score: 0.8870633560737395)]


In [31]:
west = wiki[wiki["occupation_level_3"]=="programmer"]
print(west)

        wikidata_code  gender   birth  birth_estimation  \
131         Q18684998    Male  1994.0            1994.0   
463           Q762040    Male  1968.0            1968.0   
6314         Q6141748    Male  1950.0            1950.0   
7059           Q68237    Male  1973.0            1973.0   
17023       Q16983069    Male     NaN               NaN   
...               ...     ...     ...               ...   
1272237      Q2846101    Male     NaN               NaN   
1284141      Q6307638    Male     NaN               NaN   
1287582      Q2703765    Male  1967.0            1967.0   
1295550      Q5220889    Male     NaN               NaN   
1371981        Q51769  Female  1922.0            1922.0   

                                 birth_era   death  death_estimation  \
131      5.Contemporary period 1901-2020AD     NaN       2087.801514   
463      5.Contemporary period 1901-2020AD     NaN       2055.339355   
6314     5.Contemporary period 1901-2020AD     NaN       2038.399536   
705

In [23]:
test = wiki["occupation_level_3"].unique()
print(test.tolist())

['coach', 'journalist', 'artist', 'composer', 'professor', 'football', 'lawyer', 'songwriter', 'rebel', 'cyclist', 'architect', 'education', 'radio', 'guitar', 'militar', 'politician', 'actor', 'author', 'singer', 'police', 'music', 'boxer', 'judge', 'wrestler', 'priest', 'psychologist', 'academic', 'basket', 'art_critic', 'painter', 'ski', 'drummer', 'hurler', 'theologian', 'film', 'colonel', 'sport', 'baseball', 'historian', 'cricket', 'golf', 'hockey', 'brewer', 'political', 'physicist', 'writer', 'poet', 'ballet', 'screenwriter', 'television', 'skier', 'soldier', 'pianist', 'watchmaker', 'athletic', 'entrepreneur', 'driver', 'astronomer', 'novelist', 'mayor', 'runner', 'null', 'physician', 'horse_trainer', 'programmer', 'king', 'prince', 'nun', 'handball', 'diver', 'political_scientist', 'lord', 'linguist', 'anthropologist', 'philanthropist', 'fencer', 'photographer', 'swimmer', 'polo', 'business', 'volleyball', 'chess', 'saint', 'theorist', 'emir', 'sculptor', 'player', 'banjoist'

In [29]:
all_occs = wiki["occupation_level_3"].unique().tolist()

for occ in all_occs:
    if 'developer' in occ:
        print(occ)

developer


In [116]:
occ = mapping['profession_name'].to_list()

In [125]:
sum = 0
for name in occ:
    in_wiki_num = [wiki[wiki["occupation_level_3"]==name].size,len(wiki[wiki['occupation_level_3']==name]),wiki[wiki['occupation_level_3']==name].shape[0]]
    sum += wiki[wiki['occupation_level_3']==name].shape[0]
    print(name +": " +str(in_wiki_num))
print(sum)

accountant: [5790, 193, 193]
architect: [269400, 8980, 8980]
attorney: [18330, 611, 611]
chiropractor: [0, 0, 0]
comedian: [66240, 2208, 2208]
composer: [544440, 18148, 18148]
dentist: [9270, 309, 309]
dietitian: [450, 15, 15]
dj: [3120, 104, 104]
filmmaker: [14670, 489, 489]
interior_designer: [510, 17, 17]
journalist: [553050, 18435, 18435]
model: [116250, 3875, 3875]
nurse: [23160, 772, 772]
painter: [868140, 28938, 28938]
paralegal: [60, 2, 2]
pastor: [22800, 760, 760]
personal_trainer: [0, 0, 0]
photographer: [203670, 6789, 6789]
physician: [224760, 7492, 7492]
poet: [473520, 15784, 15784]
professor: [113490, 3783, 3783]
psychologist: [89790, 2993, 2993]
rapper: [51810, 1727, 1727]
software_engineer: [0, 0, 0]
surgeon: [60150, 2005, 2005]
teacher: [65400, 2180, 2180]
yoga_teacher: [0, 0, 0]
126609


In [31]:
print(wiki_24.columns)

Index(['wikidata_code', 'gender', 'birth', 'birth_estimation', 'birth_era',
       'death', 'death_estimation', 'death_era', 'citizenship_primary',
       'citizenship_secondary', 'un_region', 'un_subregion', 'birthplace_lon',
       'birthplace_lat', 'deathplace_lon', 'deathplace_lat',
       'occupation_level_1', 'occupation_level_2', 'occupation_level_3',
       'number_wiki_editions', 'total_word_count', 'visibility_ranking',
       'visibility_sum', 'wiki_readers_2015_2018', 'curid', 'text', 'name',
       'wikipedia_url', 'title', 'id'],
      dtype='str')


In [122]:
wiki["occupation_level_3"].nunique()

2507

In [123]:
sorted(
    wiki["occupation_level_3"]
    .drop

_IncompleteInputError: incomplete input (2375101653.py, line 3)

In [118]:
bias_occs = set(
    counts[
        (counts[0] >= 100) &
        (counts[1] >= 100)
    ].index
)

wiki_occs = set(
    wiki["occupation_level_3"].unique()
)

print(sorted(bias_occs & wiki_occs))
print(len(bias_occs & wiki_occs))

['accountant', 'architect', 'attorney', 'comedian', 'composer', 'dentist', 'dietitian', 'dj', 'filmmaker', 'interior_designer', 'journalist', 'model', 'nurse', 'painter', 'paralegal', 'pastor', 'photographer', 'physician', 'poet', 'professor', 'psychologist', 'surgeon', 'teacher']
23


In [124]:
bias_occs = set(
       counts[
        (counts[0] >= 100) &
        (counts[1] >= 100)
    ].index)

wiki_occs = set(
    wiki["occupation_level_3"]
    .dropna()
    .unique()
)

overlap = sorted(bias_occs & wiki_occs)

print(len(overlap))
print(overlap)

23
['accountant', 'architect', 'attorney', 'comedian', 'composer', 'dentist', 'dietitian', 'dj', 'filmmaker', 'interior_designer', 'journalist', 'model', 'nurse', 'painter', 'paralegal', 'pastor', 'photographer', 'physician', 'poet', 'professor', 'psychologist', 'surgeon', 'teacher']


In [127]:
occupations_24 = [
    "accountant",
    "architect",
    "attorney",
    "comedian",
    "composer",
    "dentist",
    "dietitian",
    "dj",
    "filmmaker",
    "interior_designer",
    "journalist",
    "model",
    "nurse",
    "painter",
    "paralegal",
    "pastor",
    "photographer",
    "physician",
    "poet",
    "professor",
    "psychologist",
    "rapper",
    "surgeon",
    "teacher",
]

wiki_subset = wiki[
    wiki["occupation_level_3"].isin(occupations_24)
].copy()

print(wiki_subset.shape)


(126609, 30)


In [128]:
print(wiki_subset["occupation_level_3"].nunique())

24


In [129]:
wiki_subset.to_csv(
    "wiki_people_subset_24.csv",
    index=False
)

In [4]:
wiki_24 = pd.read_csv("datasets/wiki_people_subset_24.csv")

In [6]:
wiki_24.columns


Index(['wikidata_code', 'gender', 'birth', 'birth_estimation', 'birth_era',
       'death', 'death_estimation', 'death_era', 'citizenship_primary',
       'citizenship_secondary', 'un_region', 'un_subregion', 'birthplace_lon',
       'birthplace_lat', 'deathplace_lon', 'deathplace_lat',
       'occupation_level_1', 'occupation_level_2', 'occupation_level_3',
       'number_wiki_editions', 'total_word_count', 'visibility_ranking',
       'visibility_sum', 'wiki_readers_2015_2018', 'curid', 'text', 'name',
       'wikipedia_url', 'title', 'id'],
      dtype='str')

In [7]:
count2 = pd.crosstab(wiki_24['occupation_level_3'], wiki_24["gender"])

In [8]:
print(count2)

gender              Female   Male  Other
occupation_level_3                      
accountant              25    168      0
architect              645   8332      0
attorney                59    548      0
comedian               443   1761      4
composer              2005  16138      2
dentist                 44    261      2
dietitian               13      2      0
dj                      12     92      0
filmmaker              133    354      1
interior_designer        7     10      0
journalist            5224  13202      6
model                 3516    344     15
nurse                  723     49      0
painter               5220  23712      2
paralegal                1      1      0
pastor                  43    715      2
photographer          1590   5196      1
physician              977   6508      4
poet                  4035  11739      7
professor              711   3068      1
psychologist           823   2165      4
rapper                 140   1584      2
surgeon         